In [1]:
"""PTCG AI Battle | Variant C — Lucario & Hariyama Tactical Engine

Adapted from `ptcg-challenge-ultimate-sport-1000.ipynb`.
Strategy: Mega Lucario ex / Hariyama; matchup-aware heuristic policy. The included MCTS path is disabled because its search_begin call does not supply required hidden-state estimates in the current official SDK.

Run all cells to build `ptcg_ai_battle_variant_c_lucario_hariyama.tar.gz`. This notebook does not submit to Kaggle.
"""

'PTCG AI Battle | Variant C — Lucario & Hariyama Tactical Engine\n\nAdapted from `ptcg-challenge-ultimate-sport-1000.ipynb`.\nStrategy: Mega Lucario ex / Hariyama; matchup-aware heuristic policy. The included MCTS path is disabled because its search_begin call does not supply required hidden-state estimates in the current official SDK.\n\nRun all cells to build `ptcg_ai_battle_variant_c_lucario_hariyama.tar.gz`. This notebook does not submit to Kaggle.\n'

In [2]:
from pathlib import Path
import ast, gzip, hashlib, importlib.util, io, os, random, shutil, sys, tarfile, tempfile

ON_KAGGLE = Path('/kaggle/working').is_dir()
WORKDIR = Path('/kaggle/working') if ON_KAGGLE else Path(tempfile.mkdtemp(prefix='ptcg-output-'))
WORKDIR.mkdir(parents=True, exist_ok=True)
STAGING_DIR = Path(tempfile.mkdtemp(prefix='ptcg-agent-'))
MAIN_PATH = STAGING_DIR / 'main.py'
DECK_PATH = STAGING_DIR / 'deck.csv'
CG_FILES = ['__init__.py', 'api.py', 'utils.py', 'sim.py', 'libcg.so']
CG_CANDIDATES = [
    Path.cwd() / 'competitions/public-comp/ptcg-ai-battle/datasets/sample_submission/sample_submission/sample_submission/cg',
    Path.cwd() / 'datasets/sample_submission/sample_submission/sample_submission/cg',
    Path('/kaggle/input/competitions/the-pokemon-company-ptcg-ai-battle-challenge-playground/sample_submission/sample_submission/sample_submission/cg'),
]
if Path('/kaggle/input').is_dir():
    CG_CANDIDATES.extend(p.parent for p in Path('/kaggle/input').rglob('api.py') if p.parent.name == 'cg')
CG_DIR = next((p for p in CG_CANDIDATES if (p / 'api.py').is_file()), None)
if CG_DIR is None:
    raise FileNotFoundError('Official CABT cg package not found; attach the competition data.')
assert all((CG_DIR / name).is_file() for name in CG_FILES), f'Incomplete CABT package: {CG_DIR}'

# Keep intermediate files out of Kaggle Outputs; only the final archive belongs there.
os.chdir(STAGING_DIR)
sys.path.insert(0, str(CG_DIR.parent))
print('Execution:', 'Kaggle' if ON_KAGGLE else 'local check')
print('Submission output directory:', WORKDIR)
print('Private staging directory:', STAGING_DIR)
print('Official SDK:', CG_DIR)


Execution: local check
Submission output directory: /var/folders/7k/q1n29k35429d2mnrn_3mbtjr0000gn/T/ptcg-output-_d4yl4do
Private staging directory: /var/folders/7k/q1n29k35429d2mnrn_3mbtjr0000gn/T/ptcg-agent-65k1a7i6
Official SDK: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/ptcg-ai-battle/datasets/sample_submission/sample_submission/sample_submission/cg


In [3]:
MAIN_SOURCE = 'from __future__ import annotations\nimport os\nimport time\nimport math\nimport random\nimport heapq\nfrom collections import defaultdict\nfrom pathlib import Path\n\nfrom cg.api import (\n    AreaType, Card, CardType, EnergyType, Observation, OptionType,\n    Pokemon, SelectContext, all_card_data, to_observation_class,\n)\n\n_SEARCH_OK = False\ntry:\n    from cg.api import search_begin, search_step, search_end, search_release\n    _SEARCH_OK = True\nexcept Exception:\n    pass\n\nUSE_SEARCH = False  # Disabled: source calls current search_begin API with missing required hidden-state inputs.\nSEARCH_TIME_BUDGET = 1.5\nSEARCH_MAX_CANDIDATES = 8\nBEAM_WIDTH = 3\nMCTS_ITERATIONS = 15\n\nDECK = [\n    673, 673, 674, 674, 675, 675, 676, 676,\n    676, 677, 677, 677, 678, 678, 678, 678,\n    1102, 1102, 1102, 1102, 1123, 1123, 1141, 1141,\n    1141, 1141, 1142, 1142, 1142, 1142, 1152, 1152,\n    6, 1159, 1182, 1182, 1192, 1192, 1192, 1192,\n    1227, 1227, 1227, 1227, 6, 6, 6, 6,\n    6, 6, 6, 6, 6, 6, 6, 6,\n    6, 1182, 677, 1252,\n]\n\nPath("deck.csv").write_text("\\n".join(map(str, DECK)) + "\\n")\n\nclass C:\n    KYOGRE, SNOVER, MEGA_ABOMASNOW_EX = 721, 722, 723\n    MAKUHITA, HARIYAMA = 673, 674\n    LUNATONE, SOLROCK = 675, 676\n    RIOLU, MEGA_LUCARIO_EX = 677, 678\n    BASIC_FIGHTING_ENERGY = 6\n    DUSK_BALL, SWITCH, PREMIUM_POWER_PRO, FIGHTING_GONG = 1102, 1123, 1141, 1142\n    POKE_PAD, HERO_CAPE, BOSS_ORDERS = 1152, 1159, 1182\n    CARMINE, LILLIE_DETERMINATION, GRAVITY_MOUNTAIN = 1192, 1227, 1252\n    LUMIOSE_CITY, LILLIES_PEARL, LEGACY_ENERGY = 1267, 1172, 12\n\nMEGA_BRAVE = 983\nLOW_DECK_COUNT = 10\n\nDECK_PATH = "deck.csv"\nif not os.path.exists(DECK_PATH): DECK_PATH = "/kaggle_simulations/agent/deck.csv"\nwith open(DECK_PATH, "r", encoding="utf-8") as f:\n    my_deck = [int(line) for line in f.read().splitlines() if line.strip()]\n\nall_card = all_card_data()\ncard_table = {card.cardId: card for card in all_card}\n\nclass AttackPlan:\n    def __init__(self, attacker=-1, target=-1, attack_index=-1, remain_hp=-1, needs_energy=False):\n        self.attacker, self.target = attacker, target\n        self.attack_index, self.remain_hp = attack_index, remain_hp\n        self.needs_energy = needs_energy\n\nplan = AttackPlan()\npre_turn = -1\nability_used = False\n\ndef get_card(obs: Observation, area: AreaType, index: int, player_index: int) -> Pokemon | Card | None:\n    player = obs.current.players[player_index]\n    if area == AreaType.DECK: return obs.select.deck[index]\n    if area == AreaType.HAND: return player.hand[index]\n    if area == AreaType.DISCARD: return player.discard[index]\n    if area == AreaType.ACTIVE: return player.active[index]\n    if area == AreaType.BENCH: return player.bench[index]\n    if area == AreaType.PRIZE: return player.prize[index]\n    if area == AreaType.STADIUM: return obs.current.stadium[index]\n    if area == AreaType.LOOKING: return obs.current.looking[index]\n    return None\n\ndef prize_count(pokemon: Pokemon) -> int:\n    data = card_table[pokemon.id]\n    count = 3 if data.megaEx else 2 if data.ex else 1\n    for card in pokemon.energyCards:\n        if card.id == C.LEGACY_ENERGY: count -= 1\n    for card in pokemon.tools:\n        if card.id == C.LILLIES_PEARL and "Lillie" in data.name: count -= 1\n    return max(0, count)\n\ndef target_score(pokemon: Pokemon) -> int:\n    data = card_table[pokemon.id]\n    score = prize_count(pokemon) * 2000 + len(pokemon.energies) * 300 + len(pokemon.tools) * 200\n    if data.stage2: score += 500\n    elif data.stage1: score += 250\n    if pokemon.id in {144, 322, 323, 337}: score -= 200\n    if pokemon.id == C.SNOVER: score += 950\n    elif pokemon.id == C.MEGA_ABOMASNOW_EX: score += 250\n    if pokemon.id == C.RIOLU: score += 800\n    elif pokemon.id == C.MEGA_LUCARIO_EX: score += 100\n    return score + pokemon.hp\n\nclass AdvancedPolicy:\n    def __init__(self, obs: Observation):\n        self.obs = obs\n        self.state = obs.current\n        self.select = obs.select\n        self.context = self.select.context\n        self.my_index = self.state.yourIndex\n        self.op_index = 1 - self.my_index\n        self.me = self.state.players[self.my_index]\n        self.opponent = self.state.players[self.op_index]\n\n        self.field_counts = defaultdict(int)\n        self.hand_counts = defaultdict(int)\n        self.discard_counts = defaultdict(int)\n        self.has_ready_lucario_line = False\n        self.has_ready_hariyama_line = False\n        self.can_switch, self.can_gust, self.can_attack, self.can_use_mega_brave = False, False, False, False\n        self.stadium_id = self.state.stadium[0].id if self.state.stadium else 0\n\n        self._count_cards()\n        self._scan_main_options()\n\n    def choose(self) -> list[int]:\n        if not self.select.option or self.select.maxCount == 0: return []\n        if self.context == SelectContext.MAIN: self._plan_attack()\n        scores = [self._score_option(option) for option in self.select.option]\n        ranked = [i for i, _ in sorted(enumerate(scores), key=lambda item: item[1], reverse=True)]\n        self._remember_lunatone_ability(ranked)\n        return ranked[: self.select.maxCount]\n\n    def _count_cards(self) -> None:\n        for pokemon in self.me.active + self.me.bench:\n            if pokemon is None: continue\n            self.field_counts[pokemon.id] += 1\n            if pokemon.id in {C.MAKUHITA, C.HARIYAMA} and len(pokemon.energies) >= 3: self.has_ready_hariyama_line = True\n            if pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX} and len(pokemon.energies) >= 2: self.has_ready_lucario_line = True\n        for card in self.me.hand: self.hand_counts[card.id] += 1\n        for card in self.me.discard: self.discard_counts[card.id] += 1\n\n    def _scan_main_options(self) -> None:\n        if self.context != SelectContext.MAIN: return\n        for option in self.select.option:\n            if option.type == OptionType.PLAY:\n                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)\n                if card.id == C.SWITCH: self.can_switch = True\n                elif card.id == C.BOSS_ORDERS: self.can_gust = True\n            elif option.type == OptionType.EVOLVE:\n                card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)\n                if card.id == C.HARIYAMA: self.can_gust = True\n            elif option.type == OptionType.RETREAT: self.can_switch = True\n            elif option.type == OptionType.ATTACK:\n                self.can_attack = True\n                if option.attackId == MEGA_BRAVE: self.can_use_mega_brave = True\n\n    def _my_board(self) -> list[Pokemon | None]: return self.me.active + self.me.bench\n    def _opponent_board(self) -> list[Pokemon | None]: return self.opponent.active + self.opponent.bench\n    def _opponent_has(self, ids: set[int]) -> bool: return any(pokemon is not None and pokemon.id in ids for pokemon in self._opponent_board())\n    def _opponent_is_water_deck(self) -> bool: return self._opponent_has({C.KYOGRE, C.SNOVER, C.MEGA_ABOMASNOW_EX})\n    def _opponent_is_crustle_wall(self) -> bool: return self._opponent_has({344, 345})\n\n    def _can_evolve_board_index(self, board_index: int) -> bool:\n        for option in self.select.option:\n            if option.type != OptionType.EVOLVE: continue\n            target_index = option.inPlayIndex + (1 if option.inPlayArea == AreaType.BENCH else 0)\n            if target_index == board_index: return True\n        return False\n\n    def _base_attack(self, pokemon: Pokemon, attack_index: int) -> tuple[int, int, int] | None:\n        energy_required, base_damage, base_score = 0, 0, 0\n        if pokemon.id == C.MEGA_LUCARIO_EX:\n            if attack_index == 0:\n                energy_required, base_damage = 1, 130\n                base_score += 60 * min(3, self.discard_counts[C.BASIC_FIGHTING_ENERGY])\n            else:\n                energy_required, base_damage = 2, 270\n            if self._opponent_is_water_deck() and len(self.opponent.prize) <= 3: base_score -= 500\n        elif attack_index == 1: return None\n        elif pokemon.id == C.HARIYAMA: energy_required, base_damage = 3, 210\n        elif pokemon.id == C.MAKUHITA: return None\n        elif pokemon.id == C.SOLROCK and self.field_counts[C.LUNATONE] >= 1: energy_required, base_damage = 1, 70\n        if base_damage <= 0: return None\n        return energy_required, base_damage, base_score\n\n    def _base_attack_after_evolution(self, pokemon: Pokemon, board_index: int, attack_index: int):\n        if pokemon.id == C.MAKUHITA and attack_index == 0 and self._can_evolve_board_index(board_index): return 3, 210, -100\n        return self._base_attack(pokemon, attack_index)\n\n    def _plan_attack(self) -> None:\n        global plan\n        best_score = -1\n        plan = AttackPlan()\n        if self.state.turn < 2: return\n\n        for attacker_index, my_pokemon in enumerate(self._my_board()):\n            if my_pokemon is None: continue\n            if attacker_index != 0 and not self.can_switch: break\n\n            for attack_index in range(2):\n                attack = self._base_attack_after_evolution(my_pokemon, attacker_index, attack_index)\n                if attack is None: continue\n                energy_required, base_damage, base_score = attack\n                energy_count = len(my_pokemon.energies)\n                if attack_index == 1 and attacker_index == 0 and energy_count >= 2 and not self.can_use_mega_brave: break\n                needs_energy = False\n                if energy_count < energy_required:\n                    if self.hand_counts[C.BASIC_FIGHTING_ENERGY] >= 1 and not self.state.energyAttached:\n                        energy_count += 1\n                        needs_energy = energy_count >= energy_required\n                    if not needs_energy: continue\n\n                for target_index, op_pokemon in enumerate(self._opponent_board()):\n                    if op_pokemon is None: continue\n                    if target_index != 0 and not self.can_gust: break\n                    if self._opponent_is_crustle_wall() and my_pokemon.id == C.MEGA_LUCARIO_EX and op_pokemon.id == 345: continue\n\n                    damage = base_damage\n                    op_data = card_table[op_pokemon.id]\n                    if op_data.weakness == EnergyType.FIGHTING: damage *= 2\n                    elif op_data.resistance == EnergyType.FIGHTING: damage -= 30\n\n                    score = target_score(op_pokemon)\n                    prize = prize_count(op_pokemon) if op_pokemon.hp <= damage else 0\n                    if prize == 0: score *= damage / op_pokemon.hp\n                    if len(self.opponent.prize) <= prize: score = 500000\n\n                    score += base_score + (220 if attacker_index == 0 else 0) + (300 if target_index == 0 else 0) + energy_count\n                    if score > best_score:\n                        best_score = score\n                        plan = AttackPlan(attacker_index, target_index, attack_index, op_pokemon.hp - damage, needs_energy)\n\n    def _energy_target_score(self, pokemon: Pokemon, active: bool) -> int:\n        energy_count = len(pokemon.energies)\n        score = 8000 + (10 if active else 0)\n        if pokemon.id in {C.MAKUHITA, C.HARIYAMA}:\n            if pokemon.id == C.HARIYAMA: score += 1\n            if self._opponent_is_crustle_wall(): score += 260 if energy_count < 3 else 30\n            else: score += 100 if energy_count < 3 else 0; score -= 50 if self.has_ready_hariyama_line else 0\n        elif pokemon.id == C.LUNATONE: score -= 100\n        elif pokemon.id == C.SOLROCK: score += 20 if energy_count < 1 else -100\n        elif pokemon.id in {C.RIOLU, C.MEGA_LUCARIO_EX}:\n            if pokemon.id == C.MEGA_LUCARIO_EX: score += 1\n            score += 100 if energy_count < 2 else 0\n            score -= 50 if self.has_ready_lucario_line else 0\n        return score\n\n    def _score_option(self, option) -> float:\n        if option.type == OptionType.NUMBER: return option.number\n        if option.type == OptionType.YES: return 100 if self.context == SelectContext.IS_FIRST else 1\n        if option.type == OptionType.NO: return 0\n        if option.type == OptionType.CARD: return self._score_card_choice(option)\n        if option.type == OptionType.PLAY: return self._score_play(option)\n        if option.type == OptionType.ATTACH: return self._score_attach(option)\n        if option.type == OptionType.EVOLVE: return self._score_evolve(option)\n        if option.type == OptionType.ABILITY: return self._score_ability(option)\n        if option.type == OptionType.RETREAT: return 2000 if plan.attacker >= 1 else -1\n        if option.type == OptionType.ATTACK:\n            return 1100 if (option.attackId == MEGA_BRAVE) == (plan.attack_index == 1) else 1000\n        return 0\n\n    def _score_card_choice(self, option) -> float:\n        card = get_card(self.obs, option.area, option.index, option.playerIndex)\n        if card is None: return 0\n        if self.context in {SelectContext.SWITCH, SelectContext.TO_ACTIVE}: return self._score_active_choice(option, card)\n        if self.context == SelectContext.SETUP_ACTIVE_POKEMON: return 2 if card.id == C.SOLROCK and self.state.firstPlayer == self.my_index else 4 if card.id == C.SOLROCK else 3 if card.id == C.RIOLU else 1 if card.id == C.MAKUHITA else 0\n        if self.context == SelectContext.TO_HAND:\n            score = 200 - self.hand_counts[card.id] * 100\n            if card.id == C.MAKUHITA: score += (80 if self.field_counts[card.id] < 2 else -20) if self._opponent_is_crustle_wall() else (-10 if self.field_counts[card.id] >= 1 else 10)\n            elif card.id == C.HARIYAMA: score += (120 if self.field_counts[C.MAKUHITA] >= 1 else -5) if self._opponent_is_crustle_wall() else (20 if self.field_counts[C.MAKUHITA] >= 1 else -20)\n            elif card.id == C.LUNATONE: score += -250 if self.field_counts[card.id] >= 1 else 60\n            elif card.id == C.SOLROCK: score += -250 if self.field_counts[card.id] >= 1 else 50\n            elif card.id == C.RIOLU: score += -150 if (self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 2) else -3 if (self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 1) else 40\n            elif card.id == C.MEGA_LUCARIO_EX: score += 40 if self.field_counts[C.RIOLU] >= 1 else -15\n            elif card.id == C.BASIC_FIGHTING_ENERGY: score += 30 if not ability_used or not self.state.energyAttached else -1\n            return score\n        if self.context == SelectContext.ATTACH_FROM and isinstance(card, Pokemon): return self._energy_target_score(card, option.area == AreaType.ACTIVE)\n        return 0\n\n    def _score_active_choice(self, option, card: Pokemon | Card) -> float:\n        if not isinstance(card, Pokemon): return 0\n        if option.playerIndex != self.my_index: return 100 if option.index == plan.target - 1 else 0\n        score = len(card.energies) * 2\n        if option.index == plan.attacker - 1: score += 100\n        if card.id == C.MEGA_LUCARIO_EX: score += 8 if self._opponent_is_water_deck() and len(self.opponent.prize) <= 3 else 20\n        elif card.id == C.HARIYAMA and len(card.energies) >= 2: score += 45 if self._opponent_is_crustle_wall() else 15\n        elif card.id == C.MAKUHITA and len(card.energies) >= 2: score += 35 if self._opponent_is_crustle_wall() else 10\n        elif card.id == C.SOLROCK: score += 5\n        elif card.id == C.RIOLU: score += 4\n        return score\n\n    def _score_play(self, option) -> float:\n        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)\n        data = card_table[card.id]\n        if data.cardType == CardType.POKEMON:\n            if card.id in {C.LUNATONE, C.SOLROCK} and self.field_counts[card.id] >= 1: return -1\n            if card.id == C.RIOLU and self.field_counts[C.RIOLU] + self.field_counts[C.MEGA_LUCARIO_EX] >= 2: return -1\n            return 20000\n        if card.id == C.SWITCH: return 6000 if plan.attacker > 0 else -1\n        if card.id == C.PREMIUM_POWER_PRO:\n            if self.state.supporterPlayed and plan.remain_hp <= 0: return -1\n            if not self.can_attack: return 3050 if (not self.state.supporterPlayed and self.hand_counts[C.CARMINE] > 0 and self.hand_counts[C.LILLIE_DETERMINATION] == 0 and not self.me.deckCount <= LOW_DECK_COUNT) else -1\n            return 5000\n        if card.id == C.BOSS_ORDERS: return 3200 if plan.target >= 1 else -1\n        if card.id == C.CARMINE: \n            if self._opponent_is_crustle_wall() and any(c.id in {C.HARIYAMA, C.MAKUHITA} for c in self.me.hand): return -1\n            return -1 if self.me.deckCount <= LOW_DECK_COUNT else 3000\n        if card.id == C.LILLIE_DETERMINATION: \n            return -1 if self.me.deckCount <= LOW_DECK_COUNT else 3100\n        if card.id == C.GRAVITY_MOUNTAIN: return 3500 if any(p is not None and card_table[p.id].stage2 for p in self._opponent_board()) else (1200 if self.stadium_id else -1)\n        return 10000\n\n    def _score_attach(self, option) -> float:\n        card = get_card(self.obs, AreaType.HAND, option.index, self.my_index)\n        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)\n        if not isinstance(pokemon, Pokemon): return 0\n        if card.id == C.HERO_CAPE:\n            score = 7000\n            if self._opponent_is_water_deck(): return 12200 if pokemon.id == C.RIOLU else 12800 if pokemon.id == C.MEGA_LUCARIO_EX else score\n            if pokemon.id == C.RIOLU: score += 100\n            elif pokemon.id == C.MEGA_LUCARIO_EX: score += 200\n            return score\n        score = self._energy_target_score(pokemon, option.inPlayArea == AreaType.ACTIVE)\n        board_index = option.inPlayIndex if option.inPlayArea == AreaType.ACTIVE else option.inPlayIndex + 1\n        if board_index == plan.attacker and plan.needs_energy: score += 200\n        return score\n\n    def _score_evolve(self, option) -> float:\n        pokemon = get_card(self.obs, option.inPlayArea, option.inPlayIndex, self.my_index)\n        if not isinstance(pokemon, Pokemon): return 0\n        if pokemon.id == C.MAKUHITA and plan.target == 0 and not self._opponent_is_crustle_wall(): return -1\n        return 9000 + len(pokemon.energies)\n\n    def _score_ability(self, option) -> float:\n        card = get_card(self.obs, option.area, option.index, self.my_index)\n        if card.id == C.LUMIOSE_CITY: return 1\n        if card.id == C.LUNATONE and self.me.deckCount <= LOW_DECK_COUNT: return -1\n        return 30000\n\n    def _remember_lunatone_ability(self, ranked: list[int]) -> None:\n        global ability_used\n        if self.context != SelectContext.MAIN or not ranked: return\n        option = self.select.option[ranked[0]]\n        if option.type != OptionType.ABILITY: return\n        card = get_card(self.obs, option.area, option.index, self.my_index)\n        if card is not None and card.id == C.LUNATONE: ability_used = True\n\ndef evaluate_state(obs):\n    st = obs.current\n    if st is None: return 0.0\n    me, op = st.players[st.yourIndex], st.players[1 - st.yourIndex]\n    \n    prize_diff = len(op.prize) - len(me.prize)\n    if len(me.prize) == 0: return 9999999.0\n    if len(op.prize) == 0: return -9999999.0\n    val = prize_diff * 10000.0\n    \n    is_crustle = any(p is not None and p.id in {344, 345} for p in [op.active[0] if op.active else None] + list(op.bench))\n    is_snorlax = any(p is not None and p.id == 143 for p in [op.active[0] if op.active else None] + list(op.bench))\n    is_stall = is_crustle or is_snorlax\n    \n    # 1. My Board Strength\n    for p in [me.active[0] if me.active else None] + list(me.bench):\n        if p is None: continue\n        val += len(p.energies) * 200.0\n        if is_crustle:\n            if p.id == C.HARIYAMA: val += 1500.0\n            elif p.id == C.MAKUHITA: val += 800.0\n            elif p.id == C.MEGA_LUCARIO_EX: val += 0.0\n            elif p.id == C.RIOLU: val += 0.0\n        else:\n            if p.id == C.MEGA_LUCARIO_EX: val += 500.0\n            elif p.id == C.HARIYAMA: val += 300.0\n            elif p.id in {C.RIOLU, C.MAKUHITA}: val += 100.0\n            \n    # 1.5. Hand Conservation (Crucial against Stall)\n    if is_crustle:\n        for c in me.hand:\n            if c.id == C.HARIYAMA: val += 1000.0\n            elif c.id == C.MAKUHITA: val += 500.0\n        \n    if me.active and me.active[0] is not None: \n        val += me.active[0].hp * 2.0\n        if len(me.active[0].energies) >= 2: val += 500.0\n        \n    # 2. Predictive Threat Mapping (Assume opponent attaches 1 energy next turn)\n    op_max_damage = 0\n    for p in [op.active[0] if op.active else None] + list(op.bench):\n        if p is None: continue\n        val -= p.hp * 1.5\n        \n        assumed_energies = len(p.energies) + 1\n        op_dmg = 0\n        if p.id == C.MEGA_LUCARIO_EX: op_dmg = 270 if assumed_energies >= 2 else 130 if assumed_energies >= 1 else 0\n        elif p.id == C.HARIYAMA: op_dmg = 210 if assumed_energies >= 3 else 0\n        elif p.id == C.KYOGRE: op_dmg = 130 if assumed_energies >= 3 else 0\n        elif p.id == C.MEGA_ABOMASNOW_EX: op_dmg = 240 if assumed_energies >= 4 else 0\n        elif p.id == 121: op_dmg = 130 if assumed_energies >= 2 else 0 # Dragapult ex\n        else: op_dmg = assumed_energies * 40\n        op_max_damage = max(op_max_damage, op_dmg)\n        \n    # 3. Lethal Threat Penalty\n    if me.active and me.active[0] is not None:\n        my_active = me.active[0]\n        if op_max_damage >= my_active.hp:\n            prize_risk = 2 if my_active.id == C.MEGA_LUCARIO_EX else 1\n            val -= prize_risk * 4000.0\n        elif op_max_damage > 0:\n            val -= op_max_damage * 1.5\n\n    # 4. Anti-Stall Deck Conservation\n    deck_c = getattr(me, "deckCount", 60)\n    if is_stall:\n        val += deck_c * 30.0\n        val += getattr(me, "handCount", len(me.hand)) * 2.0\n    else:\n        val += getattr(me, "handCount", len(me.hand)) * 10.0\n        \n    if deck_c < 5: val -= 10000.0\n    return val\n\ndef rollout_turn(sid, cur_obs, your_index):\n    steps = 0\n    while steps < 20:\n        if cur_obs.current.result is not None and cur_obs.current.result != -1: break\n        if cur_obs.current.yourIndex != your_index: break\n        if cur_obs.select.context != SelectContext.MAIN:\n            sub = AdvancedPolicy(cur_obs).choose()\n            sel = sub[: max(1, cur_obs.select.minCount)]\n        else:\n            nxt = AdvancedPolicy(cur_obs).choose()\n            if not nxt: break\n            sel = [nxt[0]]\n            if cur_obs.select.option[nxt[0]].type == OptionType.END:\n                search_step(sid, sel)\n                break\n        ar = search_step(sid, sel)\n        if getattr(ar, "error", 0) != 0 or ar.state is None: break\n        cur_obs, sid = ar.state.observation, ar.state.searchId\n        steps += 1\n    return cur_obs\n\n\ndef simulate_action(obs, action):\n    my_p = obs.current.players[obs.current.yourIndex]\n    yd = random.sample(my_deck, getattr(my_p, "deckCount", 60))\n    sbi = search_begin(obs, your_deck=yd)\n    if getattr(sbi, "error", 0) != 0 or sbi.state is None: return -float(\'inf\')\n    ar = search_step(sbi.state.searchId, [action])\n    if getattr(ar, "error", 0) != 0 or ar.state is None: return -float(\'inf\')\n    cur = rollout_turn(ar.state.searchId, ar.state.observation, obs.current.yourIndex)\n    return evaluate_state(cur)\n\ndef SEARCH_ALGO(obs_dict, obs):\n    if not (_SEARCH_OK and USE_SEARCH): return None\n    select = obs.select\n    if select is None or select.context != SelectContext.MAIN: return None\n    t0 = time.time()\n    \n    base_order = AdvancedPolicy(obs).choose()\n    candidates = base_order[:SEARCH_MAX_CANDIDATES]\n    if not candidates: return None\n    if len(candidates) == 1: return [candidates[0]] + [i for i in base_order if i != candidates[0]]\n    \n    import math\n    visits = {a: 0 for a in candidates}\n    total_val = {a: 0.0 for a in candidates}\n    \n    try:\n        # Phase 1: Exploration\n        for a in candidates:\n            if time.time() - t0 > SEARCH_TIME_BUDGET: break\n            val = simulate_action(obs, a)\n            if val != -float(\'inf\'):\n                visits[a] += 1\n                total_val[a] += val\n                \n        # Phase 2: UCB1\n        while time.time() - t0 < SEARCH_TIME_BUDGET:\n            total_visits = sum(visits.values())\n            if total_visits == 0: break\n            \n            valid_scores = [total_val[a]/visits[a] for a in candidates if visits[a] > 0]\n            if not valid_scores: break\n            min_s = min(valid_scores)\n            max_s = max(valid_scores)\n            if max_s == min_s: max_s = min_s + 1.0\n            \n            best_ucb, best_a = -float(\'inf\'), candidates[0]\n            for a in candidates:\n                if visits[a] == 0:\n                    best_a = a\n                    break\n                avg = total_val[a] / visits[a]\n                norm_avg = (avg - min_s) / (max_s - min_s)\n                ucb = norm_avg + 0.5 * math.sqrt(math.log(total_visits) / visits[a])\n                if ucb > best_ucb:\n                    best_ucb = ucb\n                    best_a = a\n                    \n            val = simulate_action(obs, best_a)\n            if val != -float(\'inf\'):\n                visits[best_a] += 1\n                total_val[best_a] += val\n                \n        best_action = max(candidates, key=lambda a: total_val[a]/visits[a] if visits[a] > 0 else -float(\'inf\'))\n        return [best_action] + [i for i in base_order if i != best_action]\n    except Exception: return None\n\n\ndef agent(obs_dict: dict) -> list[int]:\n    try: obs = to_observation_class(obs_dict)\n    except Exception: return my_deck if obs_dict.get("select") is None else [0]\n    if obs.select is None: return my_deck\n    \n    global pre_turn, ability_used, plan\n    if pre_turn != obs.current.turn:\n        pre_turn = obs.current.turn\n        ability_used = False\n        plan = AttackPlan()\n\n    try:\n        ordered = SEARCH_ALGO(obs_dict, obs)\n        if ordered is None: ordered = AdvancedPolicy(obs).choose()\n        n = len(obs.select.option)\n        ordered = [i for i in ordered if 0 <= i < n]\n        if not ordered: return list(range(min(max(1, obs.select.minCount), n)))\n        k = max(min(obs.select.maxCount, n), min(max(1, obs.select.minCount), n))\n        return ordered[:k]\n    except Exception:\n        n = len(obs.select.option)\n        return list(range(min(max(1, obs.select.minCount), n)))\n\n'
MAIN_PATH.write_text(MAIN_SOURCE, encoding="utf-8")
DECK = [673, 673, 674, 674, 675, 675, 676, 676, 676, 677, 677, 677, 678, 678, 678, 678, 1102, 1102, 1102, 1102, 1123, 1123, 1141, 1141, 1141, 1141, 1142, 1142, 1142, 1142, 1152, 1152, 6, 1159, 1182, 1182, 1192, 1192, 1192, 1192, 1227, 1227, 1227, 1227, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1182, 677, 1252]
DECK_PATH.write_text("\n".join(map(str, DECK)) + "\n", encoding="utf-8")
assert len(DECK) == 60 and len(MAIN_SOURCE) > 10000
print("main.py:", len(MAIN_SOURCE.splitlines()), "lines")
print("deck:", len(DECK), "cards /", len(set(DECK)), "unique IDs")

main.py: 536 lines
deck: 60 cards / 17 unique IDs


In [4]:
# Compile, import the exact agent entry point and verify its initial deck response.
sys.dont_write_bytecode = True
compile(MAIN_PATH.read_text(encoding="utf-8"), str(MAIN_PATH), "exec")
spec = importlib.util.spec_from_file_location('adapted_ptcg_agent', MAIN_PATH)
policy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(policy)
assert callable(policy.agent)
assert policy.agent({'select': None, 'logs': [], 'current': None}) == DECK
from cg.api import all_card_data
cards = {card.cardId: card for card in all_card_data()}
assert all(card_id in cards for card_id in DECK), 'deck contains an unknown card ID'
counts = {}
for card_id in DECK:
    counts[card_id] = counts.get(card_id, 0) + 1
assert all(n <= 4 or cards[card_id].cardType in (5, 6) for card_id, n in counts.items()), 'non-energy card exceeds four copies'
print('Python compile, agent import, 60-card deck and card IDs: OK')


Python compile, agent import, 60-card deck and card IDs: OK


In [5]:
# Interface check only; random-opponent games are not a strength estimate.
from cg.game import battle_finish, battle_select, battle_start

def random_legal_agent(observation):
    select = observation.get('select')
    if select is None:
        return list(DECK)
    options = select.get('option') or []
    lo = int(select.get('minCount', 0))
    hi = min(int(select.get('maxCount', len(options))), len(options))
    count = random.randint(lo, hi)
    return random.sample(range(len(options)), count)

def validate_choice(observation, choice):
    select = observation['select']
    options = select.get('option') or []
    assert isinstance(choice, list) and all(isinstance(i, int) for i in choice)
    assert len(choice) == len(set(choice))
    assert all(0 <= i < len(options) for i in choice)
    assert int(select.get('minCount', 0)) <= len(choice) <= min(int(select.get('maxCount', len(options))), len(options))

def play_smoke_game(seed, policy_seat):
    random.seed(seed)
    observation, start = battle_start(DECK, DECK, reverse_player=False)
    if observation is None:
        raise RuntimeError(f'Could not start game: {start.errorPlayer=} {start.errorType=}')
    try:
        for decision in range(10000):
            state = observation.get('current') or {}
            result = int(state.get('result', -1))
            if result != -1:
                return result, decision
            if observation.get('select') is None:
                raise RuntimeError('Live game returned no legal selection.')
            seat = int(state.get('yourIndex', 0))
            choice = policy.agent(observation) if seat == policy_seat else random_legal_agent(observation)
            validate_choice(observation, choice)
            observation = battle_select(choice)
        raise RuntimeError('Game exceeded 10,000 decisions.')
    finally:
        battle_finish()

for game_number, seat in enumerate((0, 1, 0, 1), start=1):
    winner, decisions = play_smoke_game(20261005 + game_number, seat)
    print(f'game={game_number} agent_seat={seat} winner={winner} decisions={decisions}')
print('4 local games completed; these check execution and legal actions, not competitive strength.')


game=1 agent_seat=0 winner=0 decisions=123
game=2 agent_seat=1 winner=1 decisions=34
game=3 agent_seat=0 winner=0 decisions=50
game=4 agent_seat=1 winner=1 decisions=73
4 local games completed; these check execution and legal actions, not competitive strength.


In [6]:
ARCHIVE_NAME = 'ptcg_ai_battle_variant_c_lucario_hariyama.tar.gz'
for cache in STAGING_DIR.rglob('__pycache__'):
    shutil.rmtree(cache)
for bytecode in STAGING_DIR.rglob('*.pyc'):
    bytecode.unlink()
ARCHIVE = WORKDIR / ARCHIVE_NAME
with ARCHIVE.open('wb') as raw:
    with gzip.GzipFile(filename='', mode='wb', fileobj=raw, mtime=0) as zipped:
        with tarfile.open(fileobj=zipped, mode='w', format=tarfile.GNU_FORMAT) as bundle:
            for name, path in [('main.py', MAIN_PATH), ('deck.csv', DECK_PATH)] + [('cg/' + name, CG_DIR / name) for name in CG_FILES]:
                payload = path.read_bytes()
                info = tarfile.TarInfo(name)
                info.size, info.mode, info.mtime = len(payload), 0o644, 0
                info.uid = info.gid = 0
                info.uname = info.gname = ''
                bundle.addfile(info, io.BytesIO(payload))

with tarfile.open(ARCHIVE, 'r:gz') as bundle:
    expected = ['main.py', 'deck.csv', 'cg/__init__.py', 'cg/api.py', 'cg/utils.py', 'cg/sim.py', 'cg/libcg.so']
    assert bundle.getnames() == expected, bundle.getnames()
    assert bundle.extractfile('main.py').read() == MAIN_PATH.read_bytes()
    assert bundle.extractfile('deck.csv').read() == DECK_PATH.read_bytes()
print('Submission bundle:', ARCHIVE)
print('Bytes:', ARCHIVE.stat().st_size)
print('SHA-256:', hashlib.sha256(ARCHIVE.read_bytes()).hexdigest())
if ON_KAGGLE:
    from IPython.display import FileLink, display
    display(FileLink(str(ARCHIVE)))


Submission bundle: /var/folders/7k/q1n29k35429d2mnrn_3mbtjr0000gn/T/ptcg-output-_d4yl4do/ptcg_ai_battle_variant_c_lucario_hariyama.tar.gz
Bytes: 536452
SHA-256: cc9e37b7ffb174140929539f3384a589d02c329c91dff42533591b281a882d0a


In [7]:
assert ARCHIVE.is_file() and ARCHIVE.suffixes[-2:] == [".tar", ".gz"]
print("Ready for Kaggle submission: select this .tar.gz output file:")
print(ARCHIVE.name)
print("No submission was sent.")

Ready for Kaggle submission: select this .tar.gz output file:
ptcg_ai_battle_variant_c_lucario_hariyama.tar.gz
No submission was sent.
